# a05 양쪽 무릎의 좌우 대응 확인

원본 88프레임(0~87번)에서 **84번과 87번**을 판독 후보로 골랐습니다. 영상 끝에서 무릎을 다시 굽힐 때 가까운 무릎 뒤로 먼쪽 무릎 보호대의 윤곽이 보입니다. 두 장은 같은 짧은 구간에 속합니다. 30·41번의 깊게 앉은 장면에서는 먼쪽 무릎이 가려져, 여기서 얻은 결과를 그 두 장으로 옮겨 해석하기 어렵습니다.

### 1. 예측 없는 원본 확인

왼쪽·오른쪽은 **선수 자신의 방향**입니다. 메타데이터에서 카메라에 가까운 다리는 **오른쪽**입니다. 발부터 정강이와 허벅지까지 이어 보며 어느 다리인지 먼저 확인하세요.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

from wlcv.annotation.pose_points import pose_point_editor
from wlcv.config import ROOT, SPLIT, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.bilateral_knees import (
    FOLDER, FRAMES, ROI, SIDES, VIDEO_ID, draw_assignments, evaluate, read_knees,
)

meta = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id").loc[VIDEO_ID]
split = pd.read_csv(SPLIT).set_index("video_id").loc[VIDEO_ID]
if (meta.person_id, meta.near_side, split.person_id, split.role) != ("a05", "right", "a05", "validation"):
    raise ValueError("a05의 선수·가까운 방향·검증 분할을 확인하세요.")

video = VIDEOS / meta.filename
weights = ROOT / "models/yolo26n-pose.pt"
output = ROOT / "outputs/a05_bilateral_knees"
fig, axes = plt.subplots(1, len(FRAMES), figsize=(6, 5), layout="constrained")
for ax, frame in zip(axes, FRAMES):
    ax.imshow(read_rgb(video, frame))
    ax.set_title(f"a05 #{frame}")
    ax.axis("off")
plt.show()

### 2. 네 탭에서 무릎 판독

탭에 적힌 **한쪽 무릎 관절선(joint line)**을 한 번 클릭하면 원본 픽셀 좌표로 저장됩니다. 두 배로 확대한 하체 화면이며, 더 자세히 보려면 **세 배로 확대**를 선택하세요. 보호대의 색 경계와 관절선이 일치한다고 가정하지 마세요.

가려서 짚기 어려우면 **가려서 판독 불가**, 어느 다리인지 또는 위치가 불명확하면 **위치 불확실**을 누르세요. 두 경우 모두 좌표 없이 저장됩니다. 각 탭은 자기 쪽 판독만 표시합니다. 기존 30·41·60번 판독과 별도로 저장하며, 다시 열면 각 탭에서 저장한 점을 확인하거나 수정할 수 있습니다.

In [ ]:
panels, titles = [], []
for frame in FRAMES:
    for side, name in SIDES.items():
        panel = pose_point_editor(video, frame, "a05", side, split="validation",
                                  annotations=FOLDER / side, joints=("무릎",),
                                  frame_label=f"a05 {frame}번", direct_only=True, crop=ROI)
        panels.append(panel)
        titles.append(f"{frame}번 · {name}")
tabs = widgets.Tab(children=panels)
for index, title in enumerate(titles):
    tabs.set_title(index, title)
display(tabs)

### 3. 네 항목을 저장한 뒤 비교

판독한 다음 아래 셀을 실행하세요. 처음에 모든 셀을 실행했다면 이 셀만 다시 실행하면 됩니다. 원본 YOLO26n-pose로 전체 프레임을 추론합니다. 지난 실험과 동일하게 CPU, 입력 크기 640, 사람 신뢰도 0.25, 관절 신뢰도 0.50을 사용합니다.

- **같은 쪽 합**은 수동 왼쪽–예측 왼쪽과 수동 오른쪽–예측 오른쪽의 거리 합입니다.
- **교차 합**은 수동 왼쪽–예측 오른쪽과 수동 오른쪽–예측 왼쪽의 거리 합입니다.
- **같은 쪽 합 − 교차 합**이 양수이면 좌우를 바꿔 연결했을 때 총거리가 더 짧습니다. 두 교차 거리 각각이 두 같은 쪽 거리보다 모두 짧을 때만 **양쪽 모두 교차 대응 지지**로 표시합니다. 차이가 작으면 클릭 오차도 함께 고려하세요.
- 한쪽이라도 판독 불가이거나 한 사람의 양쪽 예측을 확보하지 못하면 그 프레임의 전체 대응 판정은 보류됩니다. 표에는 상태와 계산 가능한 개별 거리가 남습니다.

그림에서 **원은 수동점, ×는 YOLO**, 청록은 선수의 왼쪽, 노랑은 오른쪽입니다. 두 패널은 같은 네 점을 사용하며 연결선만 바꿉니다.

In [ ]:
manual = read_knees(FOLDER)
saved = manual["판독"].ne("미검수").sum()
print(f"무릎 판독 {saved}/{len(manual)}개 저장")
if saved < len(manual):
    display(manual[["프레임", "방향", "판독"]])
    print("미검수 탭을 저장한 뒤 이 셀을 다시 실행하세요.")
else:
    if not weights.is_file():
        raise FileNotFoundError(f"원본 자세 가중치를 확인하세요. {weights}")
    coordinates, scores = evaluate(YOLO(weights), video, manual)
    display(scores.round(2))
    display(coordinates.round(2))
    figure = draw_assignments(video, coordinates, scores)
    output.mkdir(parents=True, exist_ok=True)
    scores.to_csv(output / "knee_assignments.csv", index=False)
    coordinates.to_csv(output / "knee_coordinates.csv", index=False)
    figure.savefig(output / "knee_assignments.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"결과 저장 위치: {output}")